In [ ]:
!pip uninstall -y scikit-learn
!pip install scikit-learn==1.3.1 thop datasets baycomp

Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import json
import time
import torch
import pickle
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from math import sqrt
from tqdm import tqdm
from thop import profile
from sklearn.svm import SVR
from xgboost import XGBRegressor
from datasets import load_dataset
from baycomp import two_on_single
from itertools import combinations
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from scipy.stats import qmc, pearsonr, spearmanr
from sklearn.ensemble import RandomForestRegressor
from transformers import GPT2Config, GPT2LMHeadModel, GPT2Tokenizer
from torch.profiler import profile, record_function, ProfilerActivity
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score

# S_model

In [ ]:
dataset_path = 'drive/MyDrive/NAS/Israel/s_model/data.txt/data.txt'

dataset = []
with open(dataset_path, 'r') as file:
    lines = file.readlines()
    for line in lines:
        elements = line.split('}')
        json_string = elements[0].replace("'", '"') + '}'
        meteor_score = float(elements[1].strip('\n').replace(' ', ''))
        individual = json.loads(json_string)["individual"]

        if(individual[1] != 2048):
          dataset.append([individual, meteor_score])

len(dataset)

299

In [ ]:
# Convert dataset into structured DataFrame
df = pd.DataFrame(dataset, columns=["individual", "meteor_score"])

# Expand individual list into separate columns
df_individual = pd.DataFrame(df["individual"].tolist(), columns=[f"param_{i}" for i in range(len(df["individual"][0]))])

# Combine the expanded individual parameters with the meteor score
df_final = pd.concat([df_individual, df["meteor_score"]], axis=1)

# Display dataset
df_final

,param_0,param_1,param_2,param_3,param_4,param_5,param_6,meteor_score
0,864,1024,2,16,0.045998,0.026726,0.594053,0.361933
1,720,512,4,12,0.070958,0.193261,0.145271,0.353509
2,1280,1024,6,64,0.072702,0.119350,0.717368,0.379254
3,148,1024,48,4,0.219673,0.178750,0.142846,0.463616
4,1824,512,12,32,0.022748,0.243579,0.422617,0.369707
...,...,...,...,...,...,...,...,...
294,592,512,24,16,0.210943,0.239195,0.226623,0.360052
295,360,1024,12,12,0.170746,0.054546,0.020061,0.342867
296,784,512,6,16,0.239784,0.059268,0.869295,0.336804
297,60,1024,4,4,0.203943,0.032034,0.637028,0.469912


In [ ]:
# Separate features (X) and target variable (y)
X = df_final.drop(columns=["meteor_score"])
y = df_final["meteor_score"]

# Normalize the feature data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# Define hyperparameter grids
param_grid_svr = {
    "C": [0.1, 1, 10, 100],
    "gamma": ["scale", "auto", 0.01, 0.1, 1],
    "kernel": ["rbf", "linear", "poly"]
}

param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

param_grid_xgb = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 6, 10],
    'subsample': [0.8, 1.0]
}

param_grid_mlp = {
    'hidden_layer_sizes': [(64,), (64, 64), (128, 64)],
    'activation': ['relu', 'tanh'],
    'solver': ['adam', 'lbfgs'],
    'alpha': [0.0001, 0.001, 0.01]
}

# Define models
models = {
    "SVR": (SVR(), param_grid_svr),
    "Random Forest": (RandomForestRegressor(random_state=42), param_grid_rf),
    "XGBoost": (XGBRegressor(objective='reg:squarederror', random_state=42, verbosity=0), param_grid_xgb),
    "MLP Regressor": (MLPRegressor(max_iter=500, random_state=42), param_grid_mlp)
}

# Store results
best_models = {}

# Perform Grid Search for each model
for model_name, (model, param_grid) in models.items():
    print(f"Running Grid Search for {model_name}...")
    grid_search = GridSearchCV(model, param_grid, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1, verbose=1)
    grid_search.fit(X_scaled, y)  # Train on the full dataset

    # Store the best model
    best_models[model_name] = grid_search.best_estimator_
    print(f"Best Parameters for {model_name}: {grid_search.best_params_}")

Running Grid Search for SVR...
Fitting 3 folds for each of 60 candidates, totalling 180 fits
Best Parameters for SVR: {'C': 100, 'gamma': 'scale', 'kernel': 'linear'}
Running Grid Search for Random Forest...
Fitting 3 folds for each of 81 candidates, totalling 243 fits
Best Parameters for Random Forest: {'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 50}
Running Grid Search for XGBoost...
Fitting 3 folds for each of 54 candidates, totalling 162 fits
Best Parameters for XGBoost: {'learning_rate': 0.01, 'max_depth': 6, 'n_estimators': 100, 'subsample': 1.0}
Running Grid Search for MLP Regressor...
Fitting 3 folds for each of 36 candidates, totalling 108 fits
Best Parameters for MLP Regressor: {'activation': 'tanh', 'alpha': 0.01, 'hidden_layer_sizes': (64, 64), 'solver': 'adam'}


In [ ]:
# Number of repetitions and folds
num_repeats = 10
num_folds = 5

# Initialize dictionaries to store metrics for statistical comparison
cross_val_results = {
    "SVR": {metric: [] for metric in ["MAE", "MSE", "RMSE", "MAPE", "R² Score", "Pearson", "Spearman"]},
    "Random Forest": {metric: [] for metric in ["MAE", "MSE", "RMSE", "MAPE", "R² Score", "Pearson", "Spearman"]},
    "XGBoost": {metric: [] for metric in ["MAE", "MSE", "RMSE", "MAPE", "R² Score", "Pearson", "Spearman"]},
    "MLP Regressor": {metric: [] for metric in ["MAE", "MSE", "RMSE", "MAPE", "R² Score", "Pearson", "Spearman"]}
}

# Perform repeated k-fold cross-validation
for iteration in range(num_repeats):
    print(f"Iteration {iteration + 1}/{num_repeats}")

    kf = KFold(n_splits=num_folds, shuffle=True, random_state=iteration)

    for model_name, best_model in best_models.items():
        print(f"  Evaluating {model_name}")

        for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
            print(f"    Fold {fold + 1}/{num_folds}")

            X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            # Train model
            best_model.fit(X_train, y_train)

            # Make predictions
            y_pred = best_model.predict(X_test)

            # Compute metrics
            mae = mean_absolute_error(y_test, y_pred)
            mse = mean_squared_error(y_test, y_pred)
            rmse = sqrt(mse)
            mape = mean_absolute_percentage_error(y_test, y_pred)
            r2 = r2_score(y_test, y_pred)
            pearson_corr, _ = pearsonr(y_test, y_pred)
            spearman_corr, _ = spearmanr(y_test, y_pred)

            # Store results
            cross_val_results[model_name]["MAE"].append(mae)
            cross_val_results[model_name]["MSE"].append(mse)
            cross_val_results[model_name]["RMSE"].append(rmse)
            cross_val_results[model_name]["MAPE"].append(mape)
            cross_val_results[model_name]["R² Score"].append(r2)
            cross_val_results[model_name]["Pearson"].append(pearson_corr)
            cross_val_results[model_name]["Spearman"].append(spearman_corr)

# Compute mean and standard deviation for each metric
final_results = {}
for model_name, metrics in cross_val_results.items():
    final_results[model_name] = {metric: (np.mean(values), np.std(values)) for metric, values in metrics.items()}

# Display results
for model_name, metrics in final_results.items():
    print(f"\nModel: {model_name}")
    for metric, (mean_value, std_value) in metrics.items():
        print(f"{metric}: Mean = {mean_value:.6f}, Std = {std_value:.6f}")

Iteration 1/10
  Evaluating SVR
    Fold 1/5
    Fold 2/5
    Fold 3/5
    Fold 4/5
    Fold 5/5
  Evaluating Random Forest
    Fold 1/5
    Fold 2/5
    Fold 3/5
    Fold 4/5
    Fold 5/5
  Evaluating XGBoost
    Fold 1/5
    Fold 2/5
    Fold 3/5
    Fold 4/5
    Fold 5/5
  Evaluating MLP Regressor
    Fold 1/5
    Fold 2/5
    Fold 3/5
    Fold 4/5
    Fold 5/5
Iteration 2/10
  Evaluating SVR
    Fold 1/5
    Fold 2/5
    Fold 3/5
    Fold 4/5
    Fold 5/5
  Evaluating Random Forest
    Fold 1/5
    Fold 2/5
    Fold 3/5
    Fold 4/5
    Fold 5/5
  Evaluating XGBoost
    Fold 1/5
    Fold 2/5
    Fold 3/5
    Fold 4/5
    Fold 5/5
  Evaluating MLP Regressor
    Fold 1/5
    Fold 2/5
    Fold 3/5
    Fold 4/5
    Fold 5/5
Iteration 3/10
  Evaluating SVR
    Fold 1/5
    Fold 2/5
    Fold 3/5
    Fold 4/5
    Fold 5/5
  Evaluating Random Forest
    Fold 1/5
    Fold 2/5
    Fold 3/5
    Fold 4/5
    Fold 5/5
  Evaluating XGBoost
    Fold 1/5
    Fold 2/5
    Fold 3/5
    Fold 4/5
    

In [ ]:
# Example metrics for demonstration, replace these arrays with actual data
metrics_dict = {
    "SVR": {
        "MAE": cross_val_results["SVR"]["MAE"],
        "MSE": cross_val_results["SVR"]["MSE"],
        "RMSE": cross_val_results["SVR"]["RMSE"],
        "MAPE": cross_val_results["SVR"]["MAPE"],
        "R² Score": cross_val_results["SVR"]["R² Score"],
        "Pearson": cross_val_results["SVR"]["Pearson"],
        "Spearman": cross_val_results["SVR"]["Spearman"]
    },
    "Random Forest": {
        "MAE": cross_val_results["Random Forest"]["MAE"],
        "MSE": cross_val_results["Random Forest"]["MSE"],
        "RMSE": cross_val_results["Random Forest"]["RMSE"],
        "MAPE": cross_val_results["Random Forest"]["MAPE"],
        "R² Score": cross_val_results["Random Forest"]["R² Score"],
        "Pearson": cross_val_results["Random Forest"]["Pearson"],
        "Spearman": cross_val_results["Random Forest"]["Spearman"]
    },
    "XGBoost": {
        "MAE": cross_val_results["XGBoost"]["MAE"],
        "MSE": cross_val_results["XGBoost"]["MSE"],
        "RMSE": cross_val_results["XGBoost"]["RMSE"],
        "MAPE": cross_val_results["XGBoost"]["MAPE"],
        "R² Score": cross_val_results["XGBoost"]["R² Score"],
        "Pearson": cross_val_results["XGBoost"]["Pearson"],
        "Spearman": cross_val_results["XGBoost"]["Spearman"]
    },
    "MLP Regressor": {
        "MAE": cross_val_results["MLP Regressor"]["MAE"],
        "MSE": cross_val_results["MLP Regressor"]["MSE"],
        "RMSE": cross_val_results["MLP Regressor"]["RMSE"],
        "MAPE": cross_val_results["MLP Regressor"]["MAPE"],
        "R² Score": cross_val_results["MLP Regressor"]["R² Score"],
        "Pearson": cross_val_results["MLP Regressor"]["Pearson"],
        "Spearman": cross_val_results["MLP Regressor"]["Spearman"]
    }
}

# Ensure data is converted to numpy arrays
for model in metrics_dict:
    for metric in metrics_dict[model]:
        metrics_dict[model][metric] = np.array(metrics_dict[model][metric])

# Metrics to evaluate
metrics = ["MAE", "MSE", "RMSE", "MAPE", "R² Score", "Pearson", "Spearman"]

# Generate Bayesian Comparison and save plots
pairs = list(combinations(metrics_dict.keys(), 2))

baycomp_results = {}
for i, metric in enumerate(metrics):
    baycomp_results[metric] = {}
    for model1, model2 in pairs:
        data1 = metrics_dict[model1][metric]
        data2 = metrics_dict[model2][metric]

        # Perform Bayesian Comparison and get the figure
        (p_left, p_rope, p_right), bayes_fig = two_on_single(data1, data2, rope=0.01, plot=True)

        # Store probabilities
        baycomp_results[metric][f"{model1} vs {model2}"] = {
            "p_left": p_left,
            "p_rope": p_rope,
            "p_right": p_right
        }

        # Save the plot as a file
        filename = f"drive/MyDrive/NAS/Israel/s_model/{model1}_{model2}_{metric.replace(' ', '_')}.png"
        bayes_fig.savefig(filename)
        plt.close(bayes_fig)  # Close the figure to free memory

# Return the results for evaluation
baycomp_results

{'MAE': {'SVR vs Random Forest': {'p_left': 0.9999998905862639,
   'p_rope': 1.0941373607131766e-07,
   'p_right': 0.0},
  'SVR vs XGBoost': {'p_left': 0.9999657118256141,
   'p_rope': 3.428817438588805e-05,
   'p_right': 0.0},
  'SVR vs MLP Regressor': {'p_left': 1.0277264349130649e-11,
   'p_rope': 0.7895350030898901,
   'p_right': 0.21046499689983267},
  'Random Forest vs XGBoost': {'p_left': 8.938031846567345e-18,
   'p_rope': 0.9999999999999928,
   'p_right': 7.216449660063518e-15},
  'Random Forest vs MLP Regressor': {'p_left': 2.654595624538835e-19,
   'p_rope': 1.0951200557496321e-07,
   'p_right': 0.9999998904879944},
  'XGBoost vs MLP Regressor': {'p_left': 1.6212561448521893e-18,
   'p_rope': 6.007679240749297e-07,
   'p_right': 0.9999993992320759}},
 'MSE': {'SVR vs Random Forest': {'p_left': 6.1122702621852e-43,
   'p_rope': 1.0,
   'p_right': 0.0},
  'SVR vs XGBoost': {'p_left': 1.5119641510869243e-33,
   'p_rope': 1.0,
   'p_right': 0.0},
  'SVR vs MLP Regressor': {'p_le

In [ ]:
# Extract and process baycomp results
metrics = baycomp_results.keys()

for metric in metrics:
    # Get all comparisons for the current metric
    metric_results = baycomp_results[metric]
    model_pairs = list(metric_results.keys())

    # Initialize an empty DataFrame to hold the table
    comparison_table = pd.DataFrame(columns=["Probability A > B", "Probability A = B", "Probability A < B"], index=model_pairs)

    for pair in model_pairs:
        probs = metric_results[pair]
        comparison_table.loc[pair] = [probs['p_left'], probs['p_rope'], probs['p_right']]

    # Beautify table (optional: scale probabilities to 4 decimal places)
    comparison_table = comparison_table.map(lambda x: f"{x:.4f}")

    print(f"\nTable for Metric: {metric}")
    display(comparison_table)


Table for Metric: MAE


,Probability A > B,Probability A = B,Probability A < B
SVR vs Random Forest,1.0000,0.0000,0.0000
SVR vs XGBoost,1.0000,0.0000,0.0000
SVR vs MLP Regressor,0.0000,0.7895,0.2105
Random Forest vs XGBoost,0.0000,1.0000,0.0000
Random Forest vs MLP Regressor,0.0000,0.0000,1.0000
XGBoost vs MLP Regressor,0.0000,0.0000,1.0000



Table for Metric: MSE


,Probability A > B,Probability A = B,Probability A < B
SVR vs Random Forest,0.0000,1.0000,0.0000
SVR vs XGBoost,0.0000,1.0000,0.0000
SVR vs MLP Regressor,0.0000,1.0000,0.0000
Random Forest vs XGBoost,0.0000,1.0000,0.0000
Random Forest vs MLP Regressor,0.0000,1.0000,0.0000
XGBoost vs MLP Regressor,0.0000,1.0000,0.0000



Table for Metric: RMSE


...

,Probability A > B,Probability A = B,Probability A < B
SVR vs Random Forest,0.0025,0.0063,0.9912
SVR vs XGBoost,0.2565,0.1201,0.6233
SVR vs MLP Regressor,1.0000,0.0000,0.0000
Random Forest vs XGBoost,0.9708,0.0207,0.0085
Random Forest vs MLP Regressor,1.0000,0.0000,0.0000
XGBoost vs MLP Regressor,1.0000,0.0000,0.0000



Table for Metric: Pearson


,Probability A > B,Probability A = B,Probability A < B
SVR vs Random Forest,0.0000,0.0008,0.9992
SVR vs XGBoost,0.8566,0.1030,0.0404
SVR vs MLP Regressor,1.0000,0.0000,0.0000
Random Forest vs XGBoost,0.9998,0.0002,0.0000
Random Forest vs MLP Regressor,1.0000,0.0000,0.0000
XGBoost vs MLP Regressor,0.9993,0.0006,0.0002



Table for Metric: Spearman


,Probability A > B,Probability A = B,Probability A < B
SVR vs Random Forest,0.0337,0.1174,0.8489
SVR vs XGBoost,0.0066,0.0208,0.9726
SVR vs MLP Regressor,0.9970,0.0024,0.0006
Random Forest vs XGBoost,0.0414,0.1064,0.8522
Random Forest vs MLP Regressor,0.9993,0.0005,0.0001
XGBoost vs MLP Regressor,1.0000,0.0000,0.0000


In [ ]:
def pop_gen(pop_size):
    """
    Generate a population using hyperlatin cube sampling to maximize representation of the search space.
    """
    # Define the parameter ranges
    n_head_choices = [4, 8, 12, 16, 32, 64]  # Number of attention heads
    n_embd_choices = [
        [n_head * i for i in range(2, 72)] for n_head in n_head_choices  # Embedding dimensions per n_head
    ]
    l_choices = [512, 1024]  # Sequence length
    n_layers_choices = [2, 4, 6, 12, 24, 48]  # Number of layers

    # Dimensions: n_head, n_embd, sequence_length, num_layers, residual_dropout, attention_dropout, activation_function
    sampler = qmc.LatinHypercube(d=7)  # 7 parameters
    samples = sampler.random(n=pop_size)  # Generate normalized samples in [0, 1]

    population = []
    for sample in samples:
        n_head_idx = int(sample[0] * len(n_head_choices))
        n_head = n_head_choices[n_head_idx]

        n_embd_idx = int(sample[1] * len(n_embd_choices[n_head_idx]))
        n_embd = n_embd_choices[n_head_idx][n_embd_idx]

        l_idx = int(sample[2] * len(l_choices))
        l = l_choices[l_idx]

        n_layers_idx = int(sample[3] * len(n_layers_choices))
        n_layers = n_layers_choices[n_layers_idx]

        residual_dropout = sample[4] * 0.3  # Scale to [0, 0.3]
        attention_dropout = sample[5] * 0.3  # Scale to [0, 0.3]

        # Use raw activation value between 0.0 and 1.0 for decoding later
        activation_value = sample[6]

        individual = [
            n_embd,      # Embedding dimension
            l,           # Sequence length
            n_layers,    # Number of layers
            n_head,      # Number of attention heads
            residual_dropout,  # Residual dropout
            attention_dropout,  # Attention dropout
            activation_value,   # Raw activation function value
        ]

        population.append({'individual': individual, 'fitness': 0})  # Fitness initialized to 0

    return population

In [ ]:
# Generate a population
population = pop_gen(pop_size=5)
best_model = best_models["MLP Regressor"]

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
best_model.fit(X_train, y_train)

# Extract the randomly generated architecture
for element in population:
  new_architecture = np.array([element["individual"]])  # Convert to NumPy array

  # Scale the new architecture using the previously trained scaler
  new_architecture_scaled = scaler.transform(new_architecture)

  # Predict the Meteor Score
  predicted_score = best_model.predict(new_architecture_scaled)

  print(f'Predicted Meteor Score for individual {element["individual"]}: {predicted_score[0]}')

Predicted Meteor Score for individual [576, 512, 6, 64, 0.19116464192511548, 0.2010660024670062, 0.7239649822613254]: 0.4071872000383819
Predicted Meteor Score for individual [2016, 512, 12, 32, 0.2515804962718988, 0.2908622092648282, 0.8773472335019588]: 0.5166075773154347
Predicted Meteor Score for individual [336, 1024, 2, 12, 0.022715225181061314, 0.04887961584998572, 0.39483573090727975]: 0.3652869805346523
Predicted Meteor Score for individual [128, 1024, 48, 4, 0.09144147773151712, 0.12670050365158744, 0.44699747435947634]: 0.4087241864122506
Predicted Meteor Score for individual [588, 512, 24, 12, 0.16131382446907575, 0.07871608021244857, 0.12749129166273152]: 0.5195403881358585


/usr/local/lib/python3.11/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [ ]:
scaler_save_path = "drive/MyDrive/NAS/Israel/s_model/scaler.pkl"

# Save the scaler
with open(scaler_save_path, "wb") as file:
    pickle.dump(scaler, file)

print("✅ Scaler saved as 'scaler.pkl'")

✅ Scaler saved as 'scaler.pkl'


In [ ]:
model_save_path = "drive/MyDrive/NAS/Israel/s_model/mlp_reg.pkl"

# Save the model
with open(model_save_path, "wb") as file:
    pickle.dump(best_model, file)

print("✅ Model saved as 'mlp_reg.pkl'")

✅ Model saved as 'mlp_reg.pkl'


# FLOPs

In [ ]:
def network_decode(chromosome):
    # Extract individual from chromosome
    n_embd, sequence_length, n_layer, n_head, resid_pdrop, attn_pdrop, act_fn_value = chromosome['individual']

    # Map the activation function value to its corresponding function name
    activation_function_ranges = {
        (0.0, 0.2): "gelu",
        (0.2, 0.4): "relu",
        (0.4, 0.6): "tanh",
        (0.6, 0.8): "swish",
        (0.8, 1.0): "sigmoid"
    }

    # Decode activation function based on mapping
    for (low, high), func_name in activation_function_ranges.items():
        if low <= act_fn_value < high:
            activation_function = func_name
            break

    print(f'Activation function is {activation_function}')

    # Calculate derived parameters
    n_inner = 4 * int(n_embd)  # Explicitly cast to Python int

    # Explicitly cast all variables to Python types
    config = GPT2Config(
        vocab_size=int(50257),                   # Default GPT-2 vocabulary size
        n_positions=int(sequence_length),        # Sequence length
        n_ctx=int(sequence_length),              # Context length (same as sequence length)
        n_embd=int(n_embd),                      # Embedding dimension
        n_layer=int(n_layer),                    # Number of layers
        n_head=int(n_head),                      # Number of attention heads
        n_inner=int(n_inner),                    # Feedforward dimension
        resid_pdrop=float(resid_pdrop),          # Residual dropout
        attn_pdrop=float(attn_pdrop),            # Attention dropout
        activation_function=str(activation_function)  # Activation function
    )

    # Instantiate the model
    try:
        model = GPT2LMHeadModel(config)
    except Exception as e:
        print("Error during model creation:")
        print(e)
        raise

    return model

In [ ]:
# Load datasets
wikitext_512 = joblib.load('drive/MyDrive/NAS/Israel/datasets/wikitext2_512.pkl')
wikitext_1024 = joblib.load('drive/MyDrive/NAS/Israel/datasets/wikitext2_1024.pkl')

datasets = {
    '512': wikitext_512,
    '1024': wikitext_1024
}

In [ ]:
def compute_flops_for_text_generation(chromosome, decode_model_func, dataset_dict):
    # Decode the individual into a PyTorch model
    model = decode_model_func(chromosome)
    individual = chromosome['individual']
    model.eval()  # Set model to evaluation mode

    # Determine the sequence length and select the appropriate dataset
    sequence_length = individual[1]  # Second element is sequence length (512 or 1024)
    dataset = dataset_dict[str(sequence_length)]

    # Use only a subset for efficiency
    test_data = {
        key: dataset['test'][key][:50]  # Limit to first 50 examples for FLOPs estimation
        for key in dataset['test'].keys()
    }

    # Move model to correct device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    total_flops = 0
    with torch.no_grad():
        for _, (input_ids, attention_mask) in tqdm(
            enumerate(zip(test_data['input_ids'], test_data['attention_mask'])),
            total=len(test_data['input_ids']),
            desc="Measuring FLOPs", unit="batch"
        ):
          # Ensure input_ids are properly formatted for GPT-2
          input_ids = torch.tensor(input_ids).unsqueeze(0).to(device)  # Add batch dim
          attention_mask = torch.tensor(attention_mask).unsqueeze(0).to(device, dtype=torch.bool)

          # Try measuring FLOPs
          with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=True) as prof:
            with record_function("model_forward"):
              model(input_ids, attention_mask=attention_mask)

          # Extract FLOPs from profiler
          flops = sum([event.cpu_time for event in prof.key_averages() if "mm" in event.key])
          total_flops += flops

    estimated_flops = (total_flops / len(test_data['input_ids'])) * len(dataset['test']['input_ids'])
    print(f"Total FLOPs for text generation with sequence length {sequence_length}: {total_flops}")
    return total_flops, estimated_flops


In [ ]:
chromosome = pop_gen(pop_size=1)[0]
print(f'Chromosome is {chromosome}')

flops, est_flops = compute_flops_for_text_generation(chromosome, network_decode, datasets)

print(f'Measured FLOPs: {flops:.2f}')
print(f'Estimated FLOPs: {est_flops:.2f}')

Chromosome is {'individual': [640, 1024, 24, 16, 0.23426206888439757, 0.23719842186887113, 0.2554845822936246], 'fitness': 0}
Activation function is relu


Measuring FLOPs: 100%|██████████| 50/50 [00:32<00:00,  1.56batch/s]

Total FLOPs for text generation with sequence length 1024: 16471.774947916758
Measured FLOPs: 16471.77
Estimated FLOPs: 1435679.90
